In [ ]:
import csv
import random
import time

# Create sample TV ratings data for January to June 2026

In [ ]:
programmes = {
    "Morning News": [4.8, 4.6, 4.5, 4.2, 4.0, 3.8],
    "Drama Hour":   [6.5, 6.3, 6.1, 5.7, 5.3, 4.9],
    "Reality Show": [7.0, 7.2, 7.1, 7.4, 7.6, 7.8],
    "Game Show":    [5.5, 5.6, 5.4, 5.2, 5.0, 4.8]
}

months = ["2026-01", "2026-02", "2026-03",
          "2026-04", "2026-05", "2026-06"]

rows = []

In [ ]:
for programme, ratings in programmes.items():
    for i, month in enumerate(months):
        trp = ratings[i]
        viewers = round(trp * 1.5, 2)
        competitor = round(trp + [0.3, 0.2, 0.4, 0.7, 0.8, 1.0][i], 2)

        rows.append({
            "Month": month,
            "Programme": programme,
            "Time_Slot": {
                "Morning News": "Morning",
                "Drama Hour": "Prime Time",
                "Reality Show": "Prime Time",
                "Game Show": "Evening"
            }[programme],
            "TRP": trp,
            "Viewers_Lakh": viewers,
            "Competitor_TRP": competitor,
            "Age_Group": ["18-24", "25-34", "35-44", "45+"][i % 4],
            "Region": ["North", "South", "East", "West"][i % 4],
            "Ad_Revenue_Lakh": round(trp * 2.5, 2)
        })

Add data-quality issues for practice

In [ ]:
rows[2]["TRP"] = ""
rows[8]["TRP"] = "11.2"
rows.append(rows[5].copy())

In [ ]:
with open("tv_ratings.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
print("Total records including data-quality issues:", len(rows))
print("Fields:", list(rows[0].keys()))
print("Sample records:")
for row in rows[:5]:
    print(row)

Total records including data-quality issues: 25
Fields: ['Month', 'Programme', 'Time_Slot', 'TRP', 'Viewers_Lakh', 'Competitor_TRP', 'Age_Group', 'Region', 'Ad_Revenue_Lakh']
Sample records:
{'Month': '2026-01', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': 4.8, 'Viewers_Lakh': 7.2, 'Competitor_TRP': 5.1, 'Age_Group': '18-24', 'Region': 'North', 'Ad_Revenue_Lakh': 12.0}
{'Month': '2026-02', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': 4.6, 'Viewers_Lakh': 6.9, 'Competitor_TRP': 4.8, 'Age_Group': '25-34', 'Region': 'South', 'Ad_Revenue_Lakh': 11.5}
{'Month': '2026-03', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '', 'Viewers_Lakh': 6.75, 'Competitor_TRP': 4.9, 'Age_Group': '35-44', 'Region': 'East', 'Ad_Revenue_Lakh': 11.25}
{'Month': '2026-04', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': 4.2, 'Viewers_Lakh': 6.3, 'Competitor_TRP': 4.9, 'Age_Group': '45+', 'Region': 'West', 'Ad_Revenue_Lakh': 10.5}
{'Month': '2026-05', 'Progra

# Implementation A, without NumPy

loading and understanding the dataset

In [ ]:

with open("tv_ratings.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    raw_data = list(reader)

print("Number of records:", len(raw_data))
print("Available fields:", list(raw_data[0].keys()))
print("First five records:")
for row in raw_data[:5]:
    print(row)

print("Number of fields:", len(raw_data[0]))


Number of records: 25
Available fields: ['Month', 'Programme', 'Time_Slot', 'TRP', 'Viewers_Lakh', 'Competitor_TRP', 'Age_Group', 'Region', 'Ad_Revenue_Lakh']
First five records:
{'Month': '2026-01', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '4.8', 'Viewers_Lakh': '7.2', 'Competitor_TRP': '5.1', 'Age_Group': '18-24', 'Region': 'North', 'Ad_Revenue_Lakh': '12.0'}
{'Month': '2026-02', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '4.6', 'Viewers_Lakh': '6.9', 'Competitor_TRP': '4.8', 'Age_Group': '25-34', 'Region': 'South', 'Ad_Revenue_Lakh': '11.5'}
{'Month': '2026-03', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '', 'Viewers_Lakh': '6.75', 'Competitor_TRP': '4.9', 'Age_Group': '35-44', 'Region': 'East', 'Ad_Revenue_Lakh': '11.25'}
{'Month': '2026-04', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '4.2', 'Viewers_Lakh': '6.3', 'Competitor_TRP': '4.9', 'Age_Group': '45+', 'Region': 'West', 'Ad_Revenue_Lakh': '10.5'}
{'Month': 

Clean, deduplicate and validate

In [ ]:

clean_data = []
seen = set()
missing_count = 0
invalid_count = 0
duplicate_count = 0

for row in raw_data:
    # Remove exact duplicate rows
    key = tuple(row[field] for field in row.keys())
    if key in seen:
        duplicate_count += 1
        continue
    seen.add(key)

    # Handle missing TRP
    if row["TRP"].strip() == "":
        missing_count += 1
        continue

    # Convert and validate numeric fields
    try:
        row["TRP"] = float(row["TRP"])
        row["Viewers_Lakh"] = float(row["Viewers_Lakh"])
        row["Competitor_TRP"] = float(row["Competitor_TRP"])
        row["Ad_Revenue_Lakh"] = float(row["Ad_Revenue_Lakh"])
    except ValueError:
        invalid_count += 1
        continue

    if not 0 <= row["TRP"] <= 10:
        invalid_count += 1
        continue

    if not 0 <= row["Competitor_TRP"] <= 10:
        invalid_count += 1
        continue

    if row["Viewers_Lakh"] < 0 or row["Ad_Revenue_Lakh"] < 0:
        invalid_count += 1
        continue

    row["Programme"] = row["Programme"].strip().title()
    row["Time_Slot"] = row["Time_Slot"].strip().title()
    row["Region"] = row["Region"].strip().title()

    clean_data.append(row)

print("Original records:", len(raw_data))
print("Missing TRP records:", missing_count)
print("Invalid records:", invalid_count)
print("Duplicate records removed:", duplicate_count)
print("Clean records:", len(clean_data))


Original records: 25
Missing TRP records: 1
Invalid records: 1
Duplicate records removed: 1
Clean records: 22


# Implementation B, using NumPy

Load and understand the dataset

In [ ]:

with open("tv_ratings.csv", "r", newline="") as file:
    numpy_raw_data = list(csv.DictReader(file))

print("Number of records:", len(numpy_raw_data))
print("Available fields:", list(numpy_raw_data[0].keys()))
print("First five records:")
for row in numpy_raw_data[:5]:
    print(row)


Number of records: 25
Available fields: ['Month', 'Programme', 'Time_Slot', 'TRP', 'Viewers_Lakh', 'Competitor_TRP', 'Age_Group', 'Region', 'Ad_Revenue_Lakh']
First five records:
{'Month': '2026-01', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '4.8', 'Viewers_Lakh': '7.2', 'Competitor_TRP': '5.1', 'Age_Group': '18-24', 'Region': 'North', 'Ad_Revenue_Lakh': '12.0'}
{'Month': '2026-02', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '4.6', 'Viewers_Lakh': '6.9', 'Competitor_TRP': '4.8', 'Age_Group': '25-34', 'Region': 'South', 'Ad_Revenue_Lakh': '11.5'}
{'Month': '2026-03', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '', 'Viewers_Lakh': '6.75', 'Competitor_TRP': '4.9', 'Age_Group': '35-44', 'Region': 'East', 'Ad_Revenue_Lakh': '11.25'}
{'Month': '2026-04', 'Programme': 'Morning News', 'Time_Slot': 'Morning', 'TRP': '4.2', 'Viewers_Lakh': '6.3', 'Competitor_TRP': '4.9', 'Age_Group': '45+', 'Region': 'West', 'Ad_Revenue_Lakh': '10.5'}
{'Month': 

Clean, deduplicate and validate

In [ ]:

numpy_clean_data = []
seen_numpy = set()
missing_numpy = 0
invalid_numpy = 0
duplicate_numpy = 0

for row in numpy_raw_data:
    key = tuple(row[field] for field in row.keys())

    if key in seen_numpy:
        duplicate_numpy += 1
        continue
    seen_numpy.add(key)

    if row["TRP"].strip() == "":
        missing_numpy += 1
        continue

    try:
        trp = float(row["TRP"])
        viewers = float(row["Viewers_Lakh"])
        competitor = float(row["Competitor_TRP"])
        revenue = float(row["Ad_Revenue_Lakh"])
    except ValueError:
        invalid_numpy += 1
        continue

    if (
        not 0 <= trp <= 10
        or not 0 <= competitor <= 10
        or viewers < 0
        or revenue < 0
    ):
        invalid_numpy += 1
        continue

    row["TRP"] = trp
    row["Viewers_Lakh"] = viewers
    row["Competitor_TRP"] = competitor
    row["Ad_Revenue_Lakh"] = revenue
    row["Programme"] = row["Programme"].strip().title()
    row["Time_Slot"] = row["Time_Slot"].strip().title()
    row["Region"] = row["Region"].strip().title()

    numpy_clean_data.append(row)

print("Original records:", len(numpy_raw_data))
print("Missing TRP records:", missing_numpy)
print("Invalid records:", invalid_numpy)
print("Duplicate records removed:", duplicate_numpy)
print("Clean records:", len(numpy_clean_data))


Original records: 25
Missing TRP records: 1
Invalid records: 1
Duplicate records removed: 1
Clean records: 22
